# RAG with Groq and Llama 3 Hands-On

* Lead Professor: Júlio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>

* Assistant Professors: Fillipe Santos <a href="mailto:fillipesantos00@gmail.com">(fillipesantos00@gmail.com)</a> and Seyed Jamal Haddadi <a href="mailto:s.j.haddadi88@gmail.com">(s.j.haddadi88@gmail.com)</a>

## Description

In this session, we implement Retrieval-Augmented Generation (RAG) using Groq for inference, Pinecone as the vector database, and Llama 3 to refine responses. Our setup involves generating embeddings with HuggingFaceEncoder (dwzhu/e5-base-4k), storing and retrieving them from Pinecone, and then using Llama to enhance the retrieved text.

This notebook was created using content from [DeepLearning.ai](https://www.deeplearning.ai/), [Meta](https://about.meta.com/), and [James Briggs on YouTube](https://www.youtube.com/@jamesbriggs).


To begin, we setup our prerequisite libraries.

In [1]:
!pip install -qU datasets==2.14.5 groq pinecone-client==4.1.0 python-dotenv
!pip install -qU cohere semantic_router

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2023.6.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

## Data Preparation

We start by loading a dataset that we will encode and store. The dataset marlesson/news-of-the-site-folhauol contains news articles from the Folha and UOL websites, providing valuable content for NLP analysis in Portuguese.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from dotenv import load_dotenv
import os
import sys

# Caminho para o arquivo .env
ENV_PATH = "/content/drive/MyDrive/EXTENSÃO/2025/INF-0084 SISTEMAS INTELIGENTES E TÉCNICAS AVANÇADAS EM IA/Colabs/A3-RAG-I/.env"
#ENV_PATH = "/content/drive/MyDrive/INF-0084 SISTEMAS INTELIGENTES E TÉCNICAS AVANÇADAS EM IA/Colabs/A3-RAG-I/.env"
load_dotenv(ENV_PATH)

# Adiciona ROOT_PATH ao sys.path para imports
ROOT_PATH = "/content/drive/MyDrive/EXTENSÃO/2025/INF-0084 SISTEMAS INTELIGENTES E TÉCNICAS AVANÇADAS EM IA/Colabs/A3-RAG-I/"
#ROOT_PATH = "/content/drive/MyDrive/INF-0084 SISTEMAS INTELIGENTES E TÉCNICAS AVANÇADAS EM IA/Colabs/A3-RAG-I/"
sys.path.append(ROOT_PATH)

In [4]:
import pandas as pd
data = pd.read_csv('/content/drive/MyDrive/EXTENSÃO/2025/INF-0084 SISTEMAS INTELIGENTES E TÉCNICAS AVANÇADAS EM IA/Colabs/A3-RAG-I/articles.csv')
#data = pd.read_csv('/content/drive/MyDrive/INF-0084 SISTEMAS INTELIGENTES E TÉCNICAS AVANÇADAS EM IA/Colabs/A3-RAG-I/articles.csv')
data['id'] = range(1, len(data) + 1)
data['id'] = data['id'].astype(str)
data.head()

,title,text,date,category,subcategory,link,id
0,"Lula diz que está 'lascado', mas que ainda tem...",Com a possibilidade de uma condenação impedir ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...,1
1,"'Decidi ser escrava das mulheres que sofrem', ...","Para Oumou Sangaré, cantora e ativista malines...",2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...,2
2,Três reportagens da Folha ganham Prêmio Petrob...,Três reportagens da Folha foram vencedoras do ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...,3
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,A Disney divulgou na noite desta segunda-feira...,2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...,4
4,CBSS inicia acordos com fintechs e quer 30% do...,"O CBSS, banco da holding Elopar dos sócios Bra...",2017-09-10,mercado,NaN,http://www1.folha.uol.com.br/mercado/2017/10/1...,5


In [5]:
# Check for NaN values in the columns
if data[['title', 'text']].isna().any().any():
    print("There are NaN values in 'title' and 'text'")

    # Remove rows with NaN in the specified columns
    data = data.dropna(subset=['title', 'text'])
else:
    print("No NaN values in 'column1' and 'column2'")

There are NaN values in 'title' and 'text'


We have 200K chunks, where each chunk is roughly the length of 1-2 paragraphs in length. Here is an example of a single record:

In [43]:
data.iloc[0]

,0
title,"Lula diz que está 'lascado', mas que ainda tem..."
text,Com a possibilidade de uma condenação impedir ...
id,1
metadata,"{'id': '1', 'title': 'Lula diz que está 'lasca..."


Formating data into the format we need, this will contain `id`, `text` (which we will embed), and `metadata`.

In [7]:
data['metadata'] = data.apply(lambda x: {
    "id": x["id"],
    "title": x["title"],
    "content": x["text"]
}, axis=1)

data = data.drop(columns=[
    'date', 'category', 'subcategory', 'link'
])
data

,title,text,id,metadata
0,"Lula diz que está 'lascado', mas que ainda tem...",Com a possibilidade de uma condenação impedir ...,1,"{'id': '1', 'title': 'Lula diz que está 'lasca..."
1,"'Decidi ser escrava das mulheres que sofrem', ...","Para Oumou Sangaré, cantora e ativista malines...",2,"{'id': '2', 'title': ''Decidi ser escrava das ..."
2,Três reportagens da Folha ganham Prêmio Petrob...,Três reportagens da Folha foram vencedoras do ...,3,"{'id': '3', 'title': 'Três reportagens da Folh..."
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,A Disney divulgou na noite desta segunda-feira...,4,"{'id': '4', 'title': 'Filme 'Star Wars: Os Últ..."
4,CBSS inicia acordos com fintechs e quer 30% do...,"O CBSS, banco da holding Elopar dos sócios Bra...",5,"{'id': '5', 'title': 'CBSS inicia acordos com ..."
...,...,...,...,...
167048,"Em cenário de crise, tucano Beto Richa assume ...",O tucano Beto Richa tinha tudo para começar se...,167049,"{'id': '167049', 'title': 'Em cenário de crise..."
167049,Filho supera senador Renan Calheiros e assume ...,O economista Renan Filho (PMDB) assume nesta q...,167050,"{'id': '167050', 'title': 'Filho supera senado..."
167050,"Hoje na TV: Tottenham x Chelsea, Campeonato In...",Destaques da programação desta quinta-feira (1...,167051,"{'id': '167051', 'title': 'Hoje na TV: Tottenh..."
167051,Kim Jong-un diz estar aberto a se reunir com p...,"O líder norte-coreano, Kim Jong-un, disse nest...",167052,"{'id': '167052', 'title': 'Kim Jong-un diz est..."


##Indexing

We need to define an embedding model to create our embedding vectors for retrieval, for that we will be using a variation of the `e5-base` model with a longer context length of `4k` tokens. Ideally we should be running this on GPU for optimal runtimes.

In [8]:
from semantic_router.encoders import HuggingFaceEncoder

encoder = HuggingFaceEncoder(name="dwzhu/e5-base-4k")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/225M [00:00<?, ?B/s]

We can check whether our `encoder` will use `cpu` or a `cuda` GPU (where available).

In [45]:
encoder.device

'cuda'

We can create embeddings now like so:

In [46]:
embeds = encoder(["this is a test"])

We can view the dimensionality of our returned embeddings, which we'll need soon when initializing our vector index:

In [44]:
dims = len(embeds[0])
dims

768

We create our vector DB to store our vectors. For this we need to get a [free Pinecone API key](https://app.pinecone.io) — the API key can be found in the "API Keys" button found in the left navbar of the Pinecone dashboard.

In [12]:
#from dotenv import load_dotenv
import os
#load_dotenv()
import getpass
from pinecone import Pinecone

from google.colab import userdata
import os

api_key = os.environ.get("PINECONE") or os.getenv("PINECONE") or getpass.getpass("Enter your Pinecone API key: ")

# configure client
pc = Pinecone(api_key=api_key)

We setup our index specification. This allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [13]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

Creating an index, we set `dimension` equal to the dimensionality of our encoder (`768`), and use a `metric` also compatible with the model (this can be `cosine`). We also pass our `spec` to index initialization.

In [14]:
import time

index_name = "inf0084"
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=dims,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 200}},
 'total_vector_count': 200}

We can see the index is currently empty with a `total_vector_count` of `0`.

We begin populating it with our embeddings.

In [15]:
from tqdm.auto import tqdm

batch_size = 1  # how many embeddings we create and insert at once
data_size = 200 # how many embeddings we create in total

for i in tqdm(range(0, len(data[:data_size]), batch_size)):
    # find end of batch
    i_end = min(len(data), i+batch_size)
    # create batch
    batch = data[i:i_end]
    # create embeddings
    chunks = [f'{x["title"]}: {x["content"]}' for x in batch["metadata"]]
    embeds = encoder(chunks)
    assert len(embeds) == (i_end-i)
    to_upsert = list(zip(batch["id"], embeds, batch["metadata"]))
    # upsert to Pinecone
    index.upsert(vectors=to_upsert)

  0%|          | 0/200 [00:00<?, ?it/s]

In [28]:
batch["metadata"]

,metadata
200,"{'id': '201', 'title': ' Bairro do Paraíso t..."


##Retrieval

Now let's test retrieval!

In [17]:
def get_docs(query: str, top_k: int) -> list[str]:
    # encode query
    xq = encoder([query])
    # search pinecone index
    res = index.query(vector=xq, top_k=top_k, include_metadata=True)
    # get doc text
    docs = [x["metadata"]['content'] for x in res["matches"]]
    return docs

In [33]:
query = "França reconhecerá independência catalã?"
docs = get_docs(query, top_k=5)
print("\n---\n".join(docs))

O separatismo catalão sofreu mais um duro golpe com a entrevista da ministra francesa de relações exteriores, Nathalie Loiseau, nesta segunda-feira (9) ao canal CNews.  "Uma declaração de independência na Catalunha seria unilateral, não seria reconhecida. A primeira consequência automática seria sua saída da União Europeia", disse.  A negativa francesa sinaliza que, caso o governo catalão siga adiante e se declare independente, será isolado e pressionado por importantes atores externos. São raros os casos de apoio à Catalunha, como o do governo escocês, que tem o seu próprio movimento separatista.  Essa pressão pode dificultar os planos do presidente catalão, Carles Puigdemont, que vai discursar ao Parlamento regional em Barcelona nesta terça-feira (10) às 18h locais (às 13h em Brasília). Há receio de que ele aproveite a plenária para declarar a independência da Catalunha.  A declaração seria o resultado do plebiscito separatista de 1º de outubro, em que 90% dos eleitores votaram "sim"

Our retrieval component works, now let's try feeding this into a Llama 3 70B model hosted by Groq to produce an answer.

##Generation

Now we can generate responses using Llama 3, we'll wrap this logic into a help function called `generate`:

In [32]:
from llm_client import LLMClient
from IPython.display import Markdown, display

llm = LLMClient(model_name="llama3-70b-8192")

In [36]:
system_message = (
    "Você é um assistente especializado em responder perguntas. "
    "Sua resposta deve ser baseada exclusivamente no contexto fornecido abaixo. "
    "Se a informação solicitada não estiver presente nos documentos, responda que nenhum documento relevante foi encontrado.\n\n"
    "CONTEXTO:\n"
    + "\n---\n".join(docs)
    + "\n\nLembre-se: Se não houver um documento relevante para a pergunta, responda que nenhum documento relevante foi encontrado."
)

out = llm.predict(user_message=query, system_message=system_message)
print(out)

Segundo o texto, a estreia do documentário sobre Steven Spielberg será hoje, às 22h, na HBO.


---

In [37]:
query = "Quando será a estreia do documentario sobre Steven Spielberg?"
docs = get_docs(query, top_k=5)
print("\n---\n".join(docs))

O documentário sobre Steven Spielberg, que estreia nesta segunda (9), deveria exibir em sua primeira imagem aquela frase manjada utilizada em muitos filmes: "Baseado em fatos reais".  A trajetória do diretor é tão espetacular que pode parecer um tanto inventada, mas ali está somente a verdade. Do garoto que, aos 13, fazia ótimos filmes caseiros usando as irmãs como elenco ao magnata do cinema, produtor de 160 filmes e séries de TV.  "Spielberg", de Susan Lacy, não conseguiria dar atenção total aos 70 anos de vida e 57 de cinema de seu biografado. A narrativa se concentra em alguns de seus filmes e despreza outros. Mesmo assim, chega a duas horas e meia de duração, o que pode incomodar parte do público.  Há escolhas bem acertadas. São exibidas cenas dos seis curtas que Spielberg rodou entre os 13 e os 21 anos, nos anos 1960. A década de 1970 é muito abordada e traz os trechos mais atraentes. Mostra Spielberg em seu início na TV, dirigindo séries como "Columbo" e seu telefilme arrebatado

In [41]:
system_message = (
    "Você é um assistente especializado em responder perguntas. "
    "Sua resposta deve ser baseada exclusivamente no contexto fornecido abaixo. "
    "Se a informação solicitada não estiver presente nos documentos, responda que nenhum documento relevante foi encontrado.\n\n"
    "CONTEXTO:\n"
    + "\n---\n".join(docs)
    + "\n\nLembre-se: Se não houver um documento relevante para a pergunta, responda que nenhum documento relevante foi encontrado."
)

out = llm.predict(user_message=query, system_message=system_message)
print(out)

O investimento anunciado pela Mercedes-Benz é de R$ 2,4 bilhões no Brasil para o período entre 2018 e 2022.


---

In [47]:
query = "De quanto é o investimento anunciado pela Mercedes-Benz anuncia?"
docs = get_docs(query, top_k=5)
print("\n---\n".join(docs))

A Mercedes-Benz anunciou nesta segunda-feira (9) um investimento de R$ 2,4 bilhões no Brasil para o período entre 2018 2022.  Os recursos serão direcionados à modernização das fábricas de caminhões e chassis de ônibus da montadora em São Bernardo do Campo (SP) e Juiz de Fora (MG).  Parte dos investimentos também vão contemplar inovações nas linhas de caminhões e ônibus, segundo Philipp Schiemer, presidente da companhia no Brasil e na América Latina.  "Temos que ser competitivos a nível mundial e felizmente conseguimos a aprovação desse investimento", disse o executivo na ocasião do anúncio.  A fonte de financiamento ainda não está definida.  Segundo Schiemer, ainda há fragilidades, mas já é possível prever melhoria do cenário econômico no país.  "O câmbio se mostrou estável neste ano, apesar das turbulências políticas. Tudo isso dá confiança ao empresário. Com inflação estável, juros caindo e dólar mais ou menos estável, o empresário consegue se programar. Isso nos deixa mais animados"

In [40]:
system_message = (
    "Você é um assistente especializado em responder perguntas. "
    "Sua resposta deve ser baseada exclusivamente no contexto fornecido abaixo. "
    "Se a informação solicitada não estiver presente nos documentos, responda que nenhum documento relevante foi encontrado.\n\n"
    "CONTEXTO:\n"
    + "\n---\n".join(docs)
    + "\n\nLembre-se: Se não houver um documento relevante para a pergunta, responda que nenhum documento relevante foi encontrado."
)

out = llm.predict(user_message=query, system_message=system_message)
print(out)

O investimento anunciado pela Mercedes-Benz é de R$ 2,4 bilhões no Brasil para o período entre 2018 e 2022.


In [25]:
query = "Quando é 1+1?"
docs = get_docs(query, top_k=5)
print("\n---\n".join(docs))

Embora ninguém goste de cobrar nem de ser cobrado, fazer follow-ups frequentes com os funcionários ainda é a única forma eficaz de acompanhar o trabalho, segundo o consultor e coach executivo Silvio Celestino.  Para ele, o chefe que deseja acompanhar seus colaboradores do jeito certo deve apresentar suas expectativas e colocar prazos, além de explicar a importância das atividades em curso para o andamento do projeto.  É melhor do que passar uma lista de pendências ou estabelecer uma série de metas com números, exigindo resultados o tempo todo.  "É importante que o empreendedor saiba mostrar sua forma de pensar para a equipe, incluindo o que aprendeu durante sua trajetória profissional e até os erros que cometeu", afirma Celestino.  Reuniões diárias, de 15 minutos, são uma alternativa.  Nelas, cada colaborador deve apresentar ao chefe suas atividades da semana e o que foi feito até ali. A tática, criada pelo americano Jeff Sutherland, virou moda nas start-ups dos Estados Unidos.  Sistem

In [42]:
system_message = (
    "Você é um assistente especializado em responder perguntas. "
    "Sua resposta deve ser baseada exclusivamente no contexto fornecido abaixo. "
    "Se a informação solicitada não estiver presente nos documentos, responda que nenhum documento relevante foi encontrado.\n\n"
    "CONTEXTO:\n"
    + "\n---\n".join(docs)
    + "\n\nLembre-se: Se não houver um documento relevante para a pergunta, responda que nenhum documento relevante foi encontrado."
)


out = llm.predict(user_message=query, system_message=system_message)
print(out)

O investimento anunciado pela Mercedes-Benz é para o período entre 2018 e 2022.


Don't forget to delete your index when you're done to save resources!

In [27]:
#pc.delete_index(index_name)

---